In [227]:
import regex as re
import tiktoken

In [229]:
tokens = text.encode("utf=8")
tokens = list(map(int, tokens))

In [230]:
# my solution
new_enc = {}
def BPEncode(tokens):
    d = {}
    for pair in zip(tokens, tokens[1:]):
        d[pair] = d.get(pair, 0) + 1

    if not d or max(d.values()) == 1:
        return tokens

    max_value = max(d.values())
    key = max(d, key=d.get) 

    max_token = max(tokens) + 1
    new_enc[max_token] = key

    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == key:
            new_tokens.append(max_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return bytePairEncode(new_tokens)

In [231]:
class BasicTokenizer:
    
    def train(self, text, vocab_size, verbose=False):
        def get_stats(ids):
            counts = {}
            for pair in zip(ids, ids[1:]):
                counts[pair] = counts.get(pair, 0) + 1
            return counts
        
        def merge(ids, pair, idx):
            newids = []
            i = 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                    newids.append(idx)
                    i += 2
                else:
                    newids.append(ids[i])
                    i += 1
            return newids


        tokens = text.encode("utf=8")
        tokens = list(map(int, tokens))
        
        max_num_in_vocab = 0
        for k in tokens:
            if k > max_num_in_vocab:
                max_num_in_vocab = k
        
        
        num_merges = vocab_size - max_num_in_vocab
        ids = list(tokens)
        
        self.merges = {}
        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)
            idx = max_num_in_vocab + 1 + i
            ids = merge(ids, pair, idx)
            self.merges[pair] = idx

    def encode(self, text):
        tokens = list(text.encode("utf=8"))
        
        for pair in self.merges:
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                    new_tokens.append(self.merges.get(pair))
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens
        return tokens
        
    def decode(self, tokens):
        for key, val in reversed(self.merges.items()):
            i = 0
            new_tokens = []
            while i < len(tokens):
                if tokens[i] == val:
                    new_tokens.append(key[0])
                    new_tokens.append(key[1])
                    i += 1
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens
            
        byte_data = bytes(tokens)
        return byte_data.decode('utf-8', errors='replace')

In [298]:
class RegexTokenizer:
    def __init__(self):
        GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        self.pat = re.compile(GPT4_SPLIT_PATTERN)


    def train(self, text, vocab_size, verbose=False):
        
        def get_stats(text):
            counts = {}
            for word in text:
                for pair in zip(word, word[1:]):
                    counts[pair] = counts.get(pair, 0) + 1
            return counts
        
        def merge(word, pair, idx):
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                    new_word.append(idx)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            return new_word

        text_chunks = re.findall(self.pat, text)
        ids = [list(chunk.encode("utf-8")) for chunk in text_chunks]
        
        num_merges = vocab_size - 256
        
        self.merges = {}
        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                print("stopped at: ", i, " merges")
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            for i in range(len(ids)):
                ids[i] = merge(ids[i], pair, idx)
            self.merges[pair] = idx


    def encode(self, text):
        text_chunks = re.findall(self.pat, text)
        ids = [list(chunk.encode("utf-8")) for chunk in text_chunks]

        new_text = []
        for word in ids:
            for pair in self.merges:
                i = 0
                new_tokens = []
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                        new_tokens.append(self.merges.get(pair))
                        i += 2
                    else:
                        new_tokens.append(word[i])
                        i += 1
    
                word = new_tokens
            new_text.append(word)
        flat_tokens = [tok for word in new_text for tok in word]
        return flat_tokens
        
    def decode(self, tokens):
        for key, val in reversed(self.merges.items()):
            i = 0
            new_tokens = []
            while i < len(tokens):
                if tokens[i] == val:
                    new_tokens.append(key[0])
                    new_tokens.append(key[1])
                    i += 1
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens
            
        byte_data = bytes(tokens)
        return byte_data.decode('utf-8', errors='replace')



    def save_merges_escaped(self, filepath):
        # 1. Reconstruct the full byte representation for every token ID.
        # Start with the base 256 bytes.
        vocab = {i: bytes([i]) for i in range(256)}
        
        # Reconstruct merged tokens in chronological order
        for (p0, p1), val in self.merges.items():
            vocab[val] = vocab[p0] + vocab[p1]
            
        # 2. Now write them using the reconstructed bytes
        with open(filepath, "w", encoding="utf-8") as f:
            for (p0, p1), val in self.merges.items():
                # Retrieve the underlying bytes for p0 and p1 from our vocab
                p0_bytes = vocab[p0]
                p1_bytes = vocab[p1]
                merged_bytes = vocab[val]
                
                # Safely decode them to strings and escape control characters
                p0_text = p0_bytes.decode("utf-8", errors="replace").encode("unicode_escape").decode("utf-8")
                p1_text = p1_bytes.decode("utf-8", errors="replace").encode("unicode_escape").decode("utf-8")
                merged_text = merged_bytes.decode("utf-8", errors="replace").encode("unicode_escape").decode("utf-8")
                
                f.write(f"[{p0_text}] + [{p1_text}] -> [{merged_text}] {val}\n")

In [302]:
regEncoder = RegexTokenizer()
with open("taylorswift.txt", "r", encoding="utf-8") as f:
    text = f.read()
regEncoder.train(text, 10000)

In [303]:
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids)
text = enc.decode(ids) # get the same text back

print(text)

[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
hello world!!!? (안녕하세요!) lol123 😉


In [304]:
ids2 = regEncoder.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids2)
text2 = regEncoder.decode(ids2)

print(text2)

[5215, 1014, 5217, 293, 5231, 5232, 5233, 2859, 5237]
hello world!!!? (안녕하세요!) lol123 😉


In [305]:
regEncoder.save_merges_escaped('merges.txt')